# Loading changes: scatter and ridgeline plots

## Import packages

In [ ]:
## Time packages
import time
from datetime import datetime

## Memory packages
import psutil # tracking memory and cpu usage
import resource  # tracking memory and cpu usage
import gc
import sys

## Data structure packages
import numpy as np 
import pandas as pd # to create data frames
import pyarrow as pa

## Math and logic packages
import random
import math
from math import sqrt


## Load & Save data packages
import os
import glob
import json
import yaml
import joblib
import copy

from collections import defaultdict

## Plotting packages
import matplotlib as mpl
import matplotlib.pyplot as plt

## Packages for merging dictionaries
import os
from typing import List, Dict, Any, Iterable, Optional, Tuple, Union, Sequence
import pandas as pd
import warnings
from joblib import load as joblib_load, dump as joblib_dump

import matplotlib.cm as cm
from typing import Dict, Tuple, Optional, Sequence

from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.lines import Line2D
import zlib

## Internal functions packages
from src import figure_ops
from src import input_ops
from src import df_ops
from src import file_ops

## Define functions

In [ ]:
def plot_diff_scatter_3x2_max_with_marginals_cityagg(
    transformers_dict_city: Dict,
    lines_dict_city: Dict,
    *,
    col_b: str = "median_annual_max_loading_historical_1990_2019",
    col_f: str = "median_annual_max_loading_rcp45hotter_2030_2059",
    cities: Sequence[str] = ("AUS", "GSO", "SFO"),
    baseline_filter_col: str = "median_annual_max_loading_historical_1990_2019",
    filter_threshold_transformers: Optional[float] = None,
    filter_threshold_lines: Optional[float] = None,
    figsize: Tuple[float, float] = (12.0, 12.0),
    x_lim: Optional[Tuple[float, float]] = None,
    y_lim: Optional[Tuple[float, float]] = None,
    point_size: float = 1.0,
    point_alpha: float = 0.06,
    rasterized: bool = True,
    scatter_max_points: Optional[int] = 200_000,

    # ---------------- option to show/hide distribution axes ----------------
    show_distribution_axes: bool = True,

    show_marginals: bool = True,
    marginal_size: str = "22%",
    marginal_pad: float = 0.06,
    kde_max_points: int = 50_000,             # used for marginals + (by default) contours
    kde_grid_points: int = 256,               # used for marginals (1D KDE grid)
    kde_bandwidth: str | float = "silverman", # used for marginals + contours
    marginal_fill: bool = True,
    marginal_alpha: float = 0.25,
    marginal_lw: float = 1.2,
    clip_outliers: bool = False,
    clip_percentiles: Tuple[float, float] = (1, 99),
    city_colors: Optional[Dict[str, str]] = None,
    savepath: Optional[str] = None,

    # ---------------- Contours (2D KDE) ----------------
    show_contours: bool = True,
    contour_probs: Tuple[float, ...] = (0.50, 0.90),  # probability-mass contours
    contour_grid_points: int = 220,                   # 2D grid resolution
    contour_max_points: Optional[int] = None,         # points used for 2D KDE (defaults to kde_max_points)
    contour_bandwidth: str | float = "silverman",
    contour_scale: str = "range",                     # "range" or "std" (scales only for KDE distance)
    contour_lw: float = 1.6,
    contour_alpha: float = 0.95,
    contour_linestyles: Optional[Dict[str, str]] = None,

    # ----------------  contour smoothing control ----------------
    contour_smoothness: float = 1.0,  # higher => smoother/more oval (bandwidth multiplier)

    # ----------------  balanced, consistent scatter sampling across panels ----------------
    balance_scatter_across_cities: bool = True,        # balance color spread by equalizing points per city
    scatter_points_per_city: Optional[int] = None,     # if None, derived from scatter_max_points
    random_seed: int = 0,                               
    
    # ---------------- Loading-region label x-locations ----------------
    transformer_safe_label_x: float = 60,
    transformer_at_risk_label_x: float = 98,
    transformer_critical_label_x: float = 130,
    line_safe_label_x: float = 60,
    line_at_risk_label_x: float = 90,
    line_critical_label_x: float = 130,
):
    # ---------------- Colors ----------------
    if city_colors is None:
        # warm cities: AUS, GSO; cool city: SFO
        city_colors = {"AUS": "#A50026", "GSO": "#CC4C02", "SFO": "#2166AC"}

    if contour_linestyles is None:
        contour_linestyles = {"AUS": "-", "GSO": "-", "SFO": "-"}

    def scen_key_of(dct):
        return next(iter(dct.keys()))

    # ---------------- Data collector ----------------
    def _collect_xy(dct, city, thr, col_b, col_f):
        xs, ys = [], []
        for (_, cty), df in dct[scen_key_of(dct)].items():
            if cty != city or col_b not in df or col_f not in df:
                continue
            keep = np.ones(len(df), bool)
            if baseline_filter_col in df and thr is not None:
                keep = pd.to_numeric(df[baseline_filter_col], errors="coerce") >= thr
            b = pd.to_numeric(df[col_b], errors="coerce")[keep].to_numpy(float)
            f = pd.to_numeric(df[col_f], errors="coerce")[keep].to_numpy(float)
            x, y = f, f - b
            m = np.isfinite(x) & np.isfinite(y)
            if clip_outliers and m.any():
                lo, hi = np.nanpercentile(y[m], clip_percentiles)
                y = np.clip(y, lo, hi)
            xs.append(x[m])
            ys.append(y[m])
        if not xs:
            return np.array([]), np.array([])
        return np.concatenate(xs), np.concatenate(ys)

    panel_xy = {}
    for city in cities:
        panel_xy[(city, "Transformers")] = _collect_xy(
            transformers_dict_city, city, filter_threshold_transformers, col_b, col_f
        )
        panel_xy[(city, "Lines")] = _collect_xy(
            lines_dict_city, city, filter_threshold_lines, col_b, col_f
        )
        
        # ---------------- Summary statistics distributions ----------------
    summary_rows = []
    for city in cities:
        for asset, asset_label in (
            ("Transformers", "Transformers"),
            ("Lines", "Power lines"),
        ):
            x, y = panel_xy[(city, asset)]

            if x.size > 0:
                future_p5, future_median, future_p95 = np.nanpercentile(x, [5, 50, 95])
                change_p5, change_median, change_p95 = np.nanpercentile(y, [5, 50, 95])
            else:
                future_p5 = future_median = future_p95 = np.nan
                change_p5 = change_median = change_p95 = np.nan

            summary_rows.append({
                "City": city,
                "Asset type": asset_label,
                "N": int(x.size),
                "Future loading median [%]": future_median,
                "Future loading 5th percentile [%]": future_p5,
                "Future loading 95th percentile [%]": future_p95,
                "Change median [percentage points]": change_median,
                "Change 5th percentile [percentage points]": change_p5,
                "Change 95th percentile [percentage points]": change_p95,
            })

    summary_df = pd.DataFrame(summary_rows)    

    # ---------------- Limits ----------------
    all_x = np.concatenate([v[0] for v in panel_xy.values() if v[0].size])
    all_y = np.concatenate([v[1] for v in panel_xy.values() if v[1].size])
    xlim = x_lim or tuple(np.nanpercentile(all_x, [1, 99]))
    ylim = y_lim or tuple(np.nanpercentile(all_y, [1, 99]))

    rng = np.random.default_rng(random_seed)

    def _stable_city_seed(city: str) -> int:
        c = zlib.crc32(city.encode("utf-8")) & 0xFFFFFFFF
        return (int(random_seed) * 1000003 + int(c)) & 0xFFFFFFFF

    def subsample(x, y, n, *, rng_local=None):
        if n is None or len(x) <= n:
            return x, y
        rng_local = rng_local or rng
        idx = rng_local.choice(len(x), n, replace=False)
        return x[idx], y[idx]

    # ---------------- density-aware thinning (caps points per 2D bin) ----------------
    def cap_per_bin(x, y, xlim, ylim, nx=140, ny=140, cap=60, *, rng_local=None):
        """
        Reduce overplotting by limiting the number of points per 2D bin.
        Preserves tails while thinning the dense center.
        """
        rng_local = rng_local or rng
        x = np.asarray(x)
        y = np.asarray(y)
        m = np.isfinite(x) & np.isfinite(y)
        x = x[m]
        y = y[m]
        if x.size == 0:
            return x, y

        xr = float(xlim[1] - xlim[0])
        yr = float(ylim[1] - ylim[0])
        xr = xr if xr > 0 else 1.0
        yr = yr if yr > 0 else 1.0

        ix = ((x - xlim[0]) / xr * nx).astype(int)
        iy = ((y - ylim[0]) / yr * ny).astype(int)
        ix = np.clip(ix, 0, nx - 1)
        iy = np.clip(iy, 0, ny - 1)

        bid = ix + nx * iy
        keep_idx = []

        for b in np.unique(bid):
            idx = np.where(bid == b)[0]
            if idx.size <= cap:
                keep_idx.append(idx)
            else:
                keep_idx.append(rng_local.choice(idx, cap, replace=False))

        keep_idx = np.concatenate(keep_idx) if keep_idx else np.array([], dtype=int)
        return x[keep_idx], y[keep_idx]

    # 1D KDE for marginals (kept as-is)
    def kde_1d(x, grid, bw):
        x = x[np.isfinite(x)]
        if x.size == 0:
            return np.zeros_like(grid)
        if bw == "silverman":
            bw = 1.06 * x.std() * x.size ** (-1 / 5) if x.std() > 0 else 1.0
        z = (grid[:, None] - x[None, :]) / bw
        return np.exp(-0.5 * z**2).mean(axis=1) / (bw * np.sqrt(2 * np.pi))

    # ---------------- 2D KDE -> probability-mass contour levels ----------------
    def _scale_params_for_asset(asset: str):
        """
        Scale ONLY used for KDE distance, not for plotting.
        This avoids X dominating KDE because X range >> Y range.
        """
        xs = []
        ys = []
        for city in cities:
            x, y = panel_xy[(city, asset)]
            if x.size:
                xs.append(x)
                ys.append(y)
        if not xs:
            return 1.0, 1.0

        x_all = np.concatenate(xs)
        y_all = np.concatenate(ys)

        if contour_scale.lower() == "std":
            sx = np.nanstd(x_all)
            sy = np.nanstd(y_all)
            sx = sx if np.isfinite(sx) and sx > 0 else 1.0
            sy = sy if np.isfinite(sy) and sy > 0 else 1.0
            return sx, sy

        # default: "range" (uses axis limits)
        sx = max(float(xlim[1] - xlim[0]), 1e-9)
        sy = max(float(ylim[1] - ylim[0]), 1e-9)
        return sx, sy

    def _kde2d_on_grid(x, y, X, Y, bw, smoothness: float):
        """
        2D KDE evaluated on a grid using scipy.stats.gaussian_kde.

        smoothness multiplies the KDE bandwidth:
          - 1.0 keeps default
          - >1.0 smoother / more oval
          - <1.0 tighter / more detailed
        """
        try:
            from scipy.stats import gaussian_kde
        except Exception as e:
            raise ImportError(
                "2D KDE contours require SciPy (scipy.stats.gaussian_kde). "
            ) from e

        m = np.isfinite(x) & np.isfinite(y)
        x = x[m]
        y = y[m]
        if x.size == 0:
            return np.zeros_like(X, dtype=float)

        nmax = contour_max_points if contour_max_points is not None else kde_max_points
        if nmax is not None and x.size > nmax:
            idx = rng.choice(x.size, nmax, replace=False)
            x = x[idx]
            y = y[idx]

        # scale for KDE distance only
        sx, sy = _scale_params_for_asset(current_asset)
        xs = x / sx
        ys = y / sy

        smoothness = float(smoothness)
        if not np.isfinite(smoothness) or smoothness <= 0:
            smoothness = 1.0

        if isinstance(bw, str):
            base = bw

            def bw_method(kde_obj):
                if base == "scott":
                    return kde_obj.scott_factor() * smoothness
                if base == "silverman":
                    return kde_obj.silverman_factor() * smoothness
                return kde_obj.scott_factor() * smoothness

            kde2 = gaussian_kde(np.vstack([xs, ys]), bw_method=bw_method)
        else:
            kde2 = gaussian_kde(np.vstack([xs, ys]), bw_method=float(bw) * smoothness)

        Xs = (X / sx).ravel()
        Ys = (Y / sy).ravel()
        Z = kde2(np.vstack([Xs, Ys])).reshape(X.shape)
        return Z

    def _probability_mass_levels(Z, probs, xgrid, ygrid):
        """
        Convert probability masses (e.g., 0.5, 0.9) into density thresholds
        by approximating the integral over the grid.
        """
        Z = np.asarray(Z, float)
        if not np.isfinite(Z).any() or Z.max() <= 0:
            return []

        dx = float(xgrid[1] - xgrid[0]) if len(xgrid) > 1 else 1.0
        dy = float(ygrid[1] - ygrid[0]) if len(ygrid) > 1 else 1.0
        cell_area = dx * dy

        z = Z.ravel()
        order = np.argsort(z)[::-1]
        z_sorted = z[order]
        csum = np.cumsum(z_sorted) * cell_area
        total = csum[-1]
        if total <= 0:
            return []

        levels = []
        for p in probs:
            target = p * total
            k = np.searchsorted(csum, target, side="left")
            k = min(max(int(k), 0), len(z_sorted) - 1)
            levels.append(z_sorted[k])
        # contour() expects increasing levels
        return sorted(set(levels))

    # ---------------- Plot: NOW 1x2 (Transformers | Lines) ----------------
    fig, axes = plt.subplots(2, 1, figsize=figsize, sharex=True, sharey=True)
    axes = np.atleast_1d(axes)

    letters = iter("abcdefghijklmnopqrstuvwxyz")
    assets = ("Transformers", "Lines")

    # ---------------- NEW: same city draw order across both panels ----------------
    draw_order = list(rng.permutation(list(cities)))

    # ---------------- NEW: balanced points per city (per panel) ----------------
    if balance_scatter_across_cities:
        if scatter_points_per_city is None:
            if scatter_max_points is None:
                per_city_cap = None
            else:
                per_city_cap = max(int(scatter_max_points // max(len(cities), 1)), 1)
        else:
            per_city_cap = int(scatter_points_per_city)
    else:
        per_city_cap = scatter_max_points

    # Marginal axes are shown only if BOTH toggles are true
    use_distribution_axes = bool(show_marginals) and bool(show_distribution_axes)

    # keep contour artists if you later want interactivity (optional)
    # contour_artists = []

    for j, asset in enumerate(assets):
        ax = axes[j]

        # limits, labels, ref line
        ax.set_xlim(*xlim)
        ax.set_ylim(*ylim)
        ax.axhline(0, color="#4d4d4d", linewidth=1.0, linestyle="-", alpha=1, zorder=1000)
        
        # Rated capacity (100%) reference line
        ax.axvline(100, color="#7A0000", linewidth=1.0, linestyle="--", alpha=0.8, zorder=1000)
        
        if j==0: # transformers
            ax.axvline(80, color="#7A0000", linewidth=1.0, linestyle="--", alpha=0.8, zorder=1000)
            ax.annotate("At-risk \nloading", xy=(transformer_at_risk_label_x, 0.95), xycoords=("data", "axes fraction"), xytext=(0, 0), textcoords="offset points", rotation=0,
                va="top", ha="right", multialignment="center")
            ax.annotate("Safe \nloading", xy=(transformer_safe_label_x, 0.95), xycoords=("data", "axes fraction"), xytext=(-6, 0), textcoords="offset points", rotation=0,
                va="top", ha="right", multialignment="center")
            ax.annotate("Critical \nloading", xy=(transformer_critical_label_x, 0.95), xycoords=("data", "axes fraction"), xytext=(-6, 0), textcoords="offset points", rotation=0,
                va="top", ha="right", multialignment="center")            
        else: # lines
            ax.axvline(67, color="#7A0000", linewidth=1.0, linestyle="--", alpha=0.8, zorder=1000)
            ax.annotate("At-risk \nloading", xy=(line_at_risk_label_x, 0.95), xycoords=("data", "axes fraction"), xytext=(-1, 0), textcoords="offset points", rotation=0,
                va="top", ha="right", multialignment="center")
            ax.annotate("Safe \nloading", xy=(line_safe_label_x, 0.95), xycoords=("data", "axes fraction"), xytext=(-6, 0), textcoords="offset points", rotation=0,
                va="top", ha="right", multialignment="center")
            ax.annotate("Critical \nloading", xy=(line_critical_label_x, 0.95), xycoords=("data", "axes fraction"), xytext=(-6, 0), textcoords="offset points", rotation=0,
                va="top", ha="right", multialignment="center")

        ax.tick_params(axis="both", which="major", labelleft=True)
        ax.tick_params(axis="x", which="major", labelbottom=True)


        
        if j==0: # transformers
            ax.set_xlabel("Future maximum transformer loading [\\%]")
            ax.set_ylabel("Change in maximum \ntransformer loading [\\%]")
        else: # lines
            ax.set_xlabel("Future maximum power line loading [\\%]")
            ax.set_ylabel("Change in maximum \npower line loading [\\%]")

        # panel letter (outside, top-left)
        ax.text(
            -0.12, 1.09, next(letters),
            transform=ax.transAxes, fontweight="bold",
            va="top", ha="left"
        )

        # optional marginals (shared per panel, with all cities overlaid)
        if use_distribution_axes:
            div = make_axes_locatable(ax)
            ax_top = div.append_axes("top", size=marginal_size, pad=marginal_pad, sharex=ax)
            ax_right = div.append_axes("right", size=marginal_size, pad=marginal_pad, sharey=ax)
            for a in (ax_top, ax_right):
                a.axis("off")

            gx = np.linspace(*xlim, kde_grid_points)
            gy = np.linspace(*ylim, kde_grid_points)

        # contour grid (shared per panel)
        if show_contours:
            # safeguard: avoid contour_grid_points <= 1
            cgp = contour_grid_points if contour_grid_points is not None else 220
            if cgp < 2:
                cgp = 220
            xg = np.linspace(*xlim, cgp)
            yg = np.linspace(*ylim, cgp)
            X, Y = np.meshgrid(xg, yg)

        for city in draw_order:
            city_name = CITY_MAP.get(city, city) if "CITY_MAP" in globals() else city

            x, y = panel_xy[(city, asset)]
            if x.size == 0:
                continue

            city_rng = np.random.default_rng(_stable_city_seed(str(city)))
            xt, yt = cap_per_bin(x, y, xlim, ylim, nx=140, ny=140, cap=60, rng_local=city_rng)

            xs, ys = subsample(xt, yt, per_city_cap, rng_local=city_rng)

            ax.scatter(
                xs, ys,
                s=point_size,
                alpha=point_alpha,
                color=city_colors[city],
                marker="o",
                linewidths=0,
                edgecolors="none",
                rasterized=rasterized,
                zorder=2,
            )

            # 2D KDE contours (probability-mass levels)
            if show_contours:
                current_asset = asset  # used inside _kde2d_on_grid for scaling
                Z = _kde2d_on_grid(x, y, X, Y, contour_bandwidth, contour_smoothness)
                levels = _probability_mass_levels(Z, contour_probs, xg, yg)
                if levels:
                    ax.contour(
                        X, Y, Z,
                        levels=levels,
                        colors=[city_colors[city]],
                        linewidths=contour_lw,
                        alpha=contour_alpha,
                        linestyles=contour_linestyles.get(city, "-"),
                        zorder=3
                    )

            # marginals: overlay each city
            if use_distribution_axes:
                dx = kde_1d(x[:kde_max_points], gx, kde_bandwidth)
                dy = kde_1d(y[:kde_max_points], gy, kde_bandwidth)
                if dx.max() > 0:
                    dx /= dx.max()
                if dy.max() > 0:
                    dy /= dy.max()

                if marginal_fill:
                    ax_top.fill_between(gx, 0, dx, color=city_colors[city], alpha=marginal_alpha)
                    ax_right.fill_betweenx(gy, 0, dy, color=city_colors[city], alpha=marginal_alpha)
                ax_top.plot(gx, dx, color=city_colors[city], lw=marginal_lw)
                ax_right.plot(dy, gy, color=city_colors[city], lw=marginal_lw)

        # city legend 
        if j == 0:
            handles = []
            labels = []
            for c in cities:
                name = CITY_MAP.get(c, c) if "CITY_MAP" in globals() else c
                handles.append(
                    Line2D([0], [0], marker="o", linestyle="None",
                           markerfacecolor=city_colors[c], markeredgecolor="none",
                           markersize=6)
                )
                labels.append(name)

            fig.legend(handles, labels,
                       loc="upper center",
                       bbox_to_anchor=(0.5, 1.04),
                       ncol=len(labels),
                       frameon=True)


    plt.tight_layout()
    plt.subplots_adjust(wspace=0, hspace=0.18)

    if savepath:
        plt.savefig(savepath, dpi=600, bbox_inches="tight")
    plt.show()
    
    return summary_df

    
# ---------- KDE helpers  ----------
def _select_bandwidth(y: np.ndarray, method: str = "silverman") -> float:
    y = y[~np.isnan(y)]
    n = len(y)
    if n < 2:
        return 1.0
    std = np.std(y, ddof=1) if n > 1 else np.std(y)
    if std == 0 or np.isnan(std):
        return 1.0
    if method == "silverman":
        return 1.06 * std * n ** (-1/5)
    elif method == "scott":
        return np.power(n, -1/5) * std
    else:
        try:
            h = float(method)
            return max(h, 1e-6)
        except Exception:
            return 1.0

def _kde_gaussian(y: np.ndarray, grid: np.ndarray, bw: float) -> np.ndarray:
    y = y[~np.isnan(y)]
    if len(y) == 0:
        return np.zeros_like(grid)
    n = len(y)
    coef = 1.0 / (n * bw * sqrt(2.0 * np.pi))
    diffs = (grid[:, None] - y[None, :]) / bw
    return coef * np.exp(-0.5 * diffs**2).sum(axis=1)

def plot_diff_by_city_ridgeline_2x2_cityagg(
    transformers_dict_city: Dict,
    lines_dict_city: Dict,
    *,
    baseline_suffix: str = "historical_1990_2019",
    future_suffix: str = "rcp45hotter_2030_2059",
    cities: Sequence[str] = ("AUS", "GSO", "SFO"),
    # filtering
    baseline_filter_col: str = "median_annual_max_loading_historical_2018",
    filter_threshold_transformers: Optional[float] = None,
    filter_threshold_lines: Optional[float] = None,
    # figure/layout
    figsize: Tuple[float, float] = (12.0, 8.0),
    x_lim: Optional[Tuple[float, float] | float] = None,
    sharex: bool = True,
    # ridgeline style
    bw_method: str = "silverman",
    ridge_scale: float = 1.0,
    base_gap: float = 1.25,
    overlap_factor: float = 0.5,
    alpha_transformers: float = 0.8,
    alpha_lines: float = 0.8,
    # outlier clipping
    clip_outliers: bool = False,
    clip_percentiles: Tuple[float, float] = (1, 99),
    # per-city colors
    colors: Optional[Dict[str, str] | Sequence[str]] = None,
    # edge styles
    edgecolor_transformers: str = "#000000",
    edgecolor_lines: str = "#000000",
    lw_transformers: float = 1.1,
    lw_lines: float = 1.1,
    linestyle_transformers: str = "-",
    linestyle_lines: str = "-",
    # KDE
    grid_points: int = 256,
    # Save
    savepath: Optional[str] = None,
    display_table = True,
):
    # ---------------- Colors ----------------
    if colors is None:
        city_colors = {
            "AUS": "#0072B2",  # blue
            "GSO": "#D55E00",  # vermillion
            "SFO": "#009E73",  # bluish green
        }
    elif isinstance(colors, dict):
        city_colors = {c: colors.get(c, "#666666") for c in cities}
    else:
        palette = list(colors)
        city_colors = {c: palette[i % len(palette)] for i, c in enumerate(cities)}

    def scen_key_of(dct):
        return next(iter(dct.keys()))

    # --------------------------  CITY-LEVEL COLLECTOR --------------------------
    def _collect_city_diff_series_filtered_cityagg(dct, city, metric_key, *, row_filter_threshold):
        col_base = _METRIC_MAP[metric_key]
        col_b = f"{col_base}_{baseline_suffix}"
        col_f = f"{col_base}_{future_suffix}"

        scenario_key = scen_key_of(dct)
        out = []

        for (smartds_year, cty), df in dct[scenario_key].items():
            if cty != city:
                continue
            if col_b not in df.columns or col_f not in df.columns:
                continue

            if baseline_filter_col in df.columns and row_filter_threshold is not None:
                vals = pd.to_numeric(df[baseline_filter_col], errors="coerce").to_numpy(float)
                keep_mask = (vals >= float(row_filter_threshold))
            else:
                keep_mask = np.ones(len(df), dtype=bool)

            b = pd.to_numeric(df[col_b], errors="coerce").to_numpy(float)[keep_mask]
            f = pd.to_numeric(df[col_f], errors="coerce").to_numpy(float)[keep_mask]
            d = (f - b).astype(float)

            if clip_outliers and d.size > 0:
                low, high = np.nanpercentile(d, clip_percentiles)
                d = np.clip(d, low, high)

            if d.size > 0:
                out.append(pd.Series(d))

        if not out:
            return pd.Series([], dtype=float)

        return pd.concat(out, ignore_index=True).dropna().astype(float)

    # ---------------- Prepare panel data ----------------
    metrics_rows = ("average", "max")
    assets_cols = ("Transformers", "Lines")

    panel_data = {}
    for m in metrics_rows:
        for a in assets_cols:
            if a == "Lines":
                ser = {
                    c: _collect_city_diff_series_filtered_cityagg(
                        lines_dict_city, c, m, row_filter_threshold=filter_threshold_lines
                    )
                    for c in cities
                }
            else:
                ser = {
                    c: _collect_city_diff_series_filtered_cityagg(
                        transformers_dict_city, c, m, row_filter_threshold=filter_threshold_transformers
                    )
                    for c in cities
                }
            panel_data[(m, a)] = ser

    # ------------------------------------------------------------------
    #  Create and display summary table (median, p5, p95) for 12 dists
    # ------------------------------------------------------------------
    def _summarize_series(s: pd.Series) -> Dict[str, float]:
        """Return median, 5th, 95th percentiles for a numeric Series."""
        x = pd.to_numeric(s, errors="coerce").dropna().to_numpy(dtype=float)
        if x.size == 0:
            return {"p5": np.nan, "median": np.nan, "p95": np.nan}
        p5, med, p95 = np.nanpercentile(x, [5, 50, 95])
        return {"p5": float(p5), "median": float(med), "p95": float(p95)}

    rows = []
    for metric in metrics_rows:
        for asset in assets_cols:
            for city in cities:
                s = panel_data[(metric, asset)][city]
                stats = _summarize_series(s)
                rows.append({
                    "City": CITY_MAP.get(city, city),
                    "City_code": city,
                    "Asset": asset,
                    "Metric": metric,  # "average" or "max"
                    "p5": stats["p5"],
                    "median": stats["median"],
                    "p95": stats["p95"],
                })

    df_summary = pd.DataFrame(rows)

    #  ordering + formatting
    df_summary["Metric"] = df_summary["Metric"].map({"average": "Δ average", "max": "Δ max"})
    df_summary = df_summary[["City", "Asset", "Metric", "p5", "median", "p95"]]

    # optional: sort
    df_summary = df_summary.sort_values(["Asset", "Metric", "City"]).reset_index(drop=True)

    # display
    if display_table:
        try:
            from IPython.display import display  # type: ignore
            display(df_summary.style.format({"p5": "{:.2f}", "median": "{:.2f}", "p95": "{:.2f}"}))
        except Exception:
            print(df_summary.to_string(index=False))

    # ---------------- Global x-limits ----------------
    if x_lim is None and sharex:
        stacks = []
        for key in panel_data:
            for c in cities:
                s = panel_data[key][c]
                if len(s):
                    stacks.append(s)
        if stacks:
            full = pd.concat(stacks, ignore_index=True)
            vmin, vmax = np.nanpercentile(full, [1, 99])
            pad = 0.08 * max(vmax - vmin, 1.0)
            x_lim = (vmin - pad, vmax + pad)

    def apply_xlim(ax_):
        if x_lim is None:
            return
        if isinstance(x_lim, (tuple, list)) and len(x_lim) == 2:
            ax_.set_xlim(x_lim[0], x_lim[1])
        else:
            xmin = ax_.get_xlim()[0]
            ax_.set_xlim(xmin, float(x_lim))

    # ------------------- Figure -------------------
    fig, axes = plt.subplots(2, 2, figsize=figsize, sharex=sharex)
    letter_labels = {(0,0): "a", (0,1): "b", (1,0): "c", (1,1): "d"}

    # ------------------- Draw ridgelines -------------------
    def draw_ridgeline_panel(ax, series_by_city, asset_type: str):
        true_gap = base_gap * (1 - np.clip(overlap_factor, 0, 1))
        y_positions = [i * true_gap for i in range(len(cities))]

        if isinstance(x_lim, (tuple, list)) and len(x_lim) == 2:
            vmin_g, vmax_g = x_lim
        else:
            stacks = [series_by_city[c] for c in cities if len(series_by_city[c])]
            if stacks:
                full = pd.concat(stacks, ignore_index=True)
                vmin_g, vmax_g = (np.nanmin(full), np.nanmax(full))
            else:
                vmin_g, vmax_g = -1.0, 1.0
        grid = np.linspace(vmin_g, vmax_g, grid_points)

        for yi, c in zip(y_positions, cities):
            s = series_by_city[c].to_numpy(float)
            n = len(s)
            color = city_colors[c]

            if asset_type == "Transformers":
                alpha = alpha_transformers; ecolor = edgecolor_transformers
                lw = lw_transformers; lstyle = linestyle_transformers
            else:
                alpha = alpha_lines; ecolor = edgecolor_lines
                lw = lw_lines; lstyle = linestyle_lines

            if n == 0:
                ax.text(grid[0], yi + 0.06, CITY_MAP.get(c, c),
                        va="bottom", ha="left")
                continue

            bw = _select_bandwidth(s, bw_method)
            dens = _kde_gaussian(s, grid, bw)
            if dens.max() > 0:
                dens = dens / dens.max() * ridge_scale

            ax.fill_between(grid, yi, yi + dens, color=color, alpha=alpha)
            ax.plot(grid, yi + dens, color=ecolor, linewidth=lw, linestyle=lstyle)
            ax.text(grid[0], yi + 0.06, CITY_MAP.get(c, c),
                    va="bottom", ha="left")

        # add vertical line at zero    
        ax.axvline(0.0, color="black", linewidth=1.0, linestyle="--", alpha=0.8, zorder=0)

        ax.set_yticks([])
        ax.grid(axis="x", alpha=0.6)
        apply_xlim(ax)

    for r, metric_key in enumerate(metrics_rows):
        for ci, asset_type in enumerate(assets_cols):
            ax = axes[r, ci]
            draw_ridgeline_panel(ax, panel_data[(metric_key, asset_type)], asset_type)
            ax.text(0.02, 0.96, letter_labels[(r, ci)],
                    transform=ax.transAxes, ha="left", va="top", fontweight="bold")
            if r == 0:
                ax.set_xlabel("Change in mean loading [\%]")
            if r == 1:
                ax.set_xlabel("Change in median maximum loading [\%]")
                
            # Y-axis labels by column
            if ci == 0:
                ax.set_ylabel("Distribution across transformers")
            elif ci == 1:
                ax.set_ylabel("Distribution across power lines")

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.25)

    if savepath:
        plt.savefig(savepath, bbox_inches="tight", pad_inches=0.5, dpi=600)
    plt.show()
    
    
def count_assets_by_city(data_dict, cities):
    """Count rows/assets across all DataFrames belonging to each city."""
    
    scenario_key = next(iter(data_dict.keys()))
    inner_dict = data_dict[scenario_key]

    counts = {}

    for city in cities:
        n_assets = 0

        for key, df in inner_dict.items():
            # Works whether keys are e.g. ('2018', 'AUS')
            # or ('2018', 'AUS', 'P1R')
            if city in key:
                n_assets += len(df)

        counts[city] = n_assets

    return counts

## Load config file with scenarios and parameters 

In [ ]:
config_file_name = 'opendss_config1'; config_path = f"config/{config_file_name}.yaml"; config = input_ops.load_config(config_path)
# pprint.pprint(config, sort_dicts=False) # print config

smart_ds_year = config['smart_ds_years'][0]

## Initialize parameters for saving paths
output_pf_path = config['output_pf_path']

# Percent-of-peak range to load (e.g., top 0–10% hours)
start_row_percent = config['start_row_percent']
top_percent_mdh = config['top_percent_mdh']

# File names
transformers_file_name = f"transformers_top_{start_row_percent}_{top_percent_mdh}_percent"
lines_file_name = f"lines_top_{start_row_percent}_{top_percent_mdh}_percent"

top_n_hours = int(np.ceil(8760*top_percent_mdh/100)) # calculate top city demand hours to run (top_percent_mdh% of hours of the year)

# Solar / battery SMART-DS scenario parameters
solar_share = config.get("solar_share", "none")
battery_share = config.get("battery_share", "none")

solar_battery_scenario_folder = input_ops.build_solar_battery_scenario_folder(
    solar_share=solar_share,
    battery_share=battery_share,
)

print(f"solar_share: {solar_share}\n battery_share: {battery_share}\n solar_battery_scenario_folder: {solar_battery_scenario_folder}")


print(f"\n\nsmart_ds_year: {smart_ds_year} \n\nstart_row_percent: {start_row_percent} \n\ntop_percent_mdh: {top_percent_mdh}  \n\nOutput_pf_path: {output_pf_path}")

## Set parameters

In [ ]:
xfer_at_risk_threshold = 80
lines_at_risk_threshold = 67

at_risk_filter_column = 'median_annual_max_loading_historical_1990_2019' # to filter population at-risk, e.g., max_loading_historical_2018 or median_annual_max_loading_historical_1990_2019

save_folder = "all_regions"

TGW_scenario = "rcp45hotter"

hist_TGW_weather_year = '1990_2019'
fut_TGW_weather_year = '2030_2059'

fig_scatter_name = f'figures/loading_change/loading_distribution/scatter_contour_1x2_max_with_marginals_by_city_{fut_TGW_weather_year}_{save_folder}.pdf'

fig_scatter_name_max_full_pop = f'figures/loading_change/loading_distribution/scatter_contour_1x2_max_with_marginals_by_city_{fut_TGW_weather_year}_{save_folder}_full_pop.pdf'
fig_scatter_name_max_at_risk_pop = f'figures/loading_change/loading_distribution/scatter_contour_1x2_max_with_marginals_by_city_{fut_TGW_weather_year}_{save_folder}_at_risk_pop.pdf'

# Create the directory structure if it doesn't exist
os.makedirs(os.path.dirname(fig_scatter_name_max_full_pop), exist_ok=True)

# suffix for ridgeplot
baseline_suffix = "historical_1990_2019"
future_suffix = f"{TGW_scenario}_{fut_TGW_weather_year}"


cities = ["AUS","GSO","SFO"]

CITY_REGIONS_TO_RUN = {
    "GSO": ["rural", "industrial", "urban-suburban"],
    "SFO": ["P1U", "P2U", "P1R"],
    "AUS": ["P1U", "P1R", "P2U"],
}

CITY_MAP = {"AUS": "Austin", "GSO": "Greensboro", "SFO": "San-Francisco"}


# ---------- Mapping for ridge plot ----------
_METRIC_MAP = {
    "max": "median_annual_max_loading",
    "min": "median_annual_min_loading",
    "average": "median_annual_average_loading",
    "90th": "median_annual_90th_loading",
    "95th": "median_annual_95th_loading",
    "99th": "median_annual_99th_loading",
}

# city_colors={"AUS": "#CC4C02", "GSO": "#2166AC", "SFO": "#A50026"}
city_colors={"AUS": "#D55E00", "GSO": "#009E73", "SFO": "#56B4E9"}  

## Parameters for figures format
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]
mpl.rcParams['text.usetex'] = True

## Load merged dictionary w/ summary statistics across years

In [ ]:
# ============================================================
# Load period-level summary dictionaries
# ============================================================
summary_save_dir = os.path.join(
    output_pf_path,
    save_folder,
    "summary_across_weather_years",
    TGW_scenario,
    solar_battery_scenario_folder,
)
transformers_summary_across_years_path = os.path.join(
    summary_save_dir,
    f"{transformers_file_name}_summary_across_weather_years.joblib",
)

lines_summary_across_years_path = os.path.join(
    summary_save_dir,
    f"{lines_file_name}_summary_across_weather_years.joblib",
)

# Check files exist before loading
for path in [
    transformers_summary_across_years_path,
    lines_summary_across_years_path,
]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

summary_transformers_dict_weather = joblib.load(
    transformers_summary_across_years_path
)

summary_lines_dict_weather = joblib.load(
    lines_summary_across_years_path
)

print("Loaded:")
print(transformers_summary_across_years_path)
print(lines_summary_across_years_path)

# ============================================================
# Inspect loaded dictionaries
# ============================================================

print("\nTransformer summary dictionary nested keys:")
file_ops.print_nested_keys_structure(summary_transformers_dict_weather)

print("\nTransformer summary dictionary sample dataframe:")
file_ops.print_nested_dict_key_examples_and_dataframe_details(
    summary_transformers_dict_weather
)

## Process data (merge regions, add mean and max diff)

In [ ]:
### Concatenate regions to single city level dataframes (e.g., a single df with all transformers in GSO)
transformers_dict_summary_multi_region_agg_by_city, lines_dict_summary_multi_region_agg_by_city = df_ops.concat_regions_to_city(summary_transformers_dict_weather, summary_lines_dict_weather, fut_TGW_weather_year, TGW_scenario, smart_ds_year, CITY_REGIONS_TO_RUN)

for city in cities:
    # Add column with difference in average loading 
    transformers_dict_summary_multi_region_agg_by_city[(fut_TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]['diff_in_median_loading_future_minus_baseline'] = transformers_dict_summary_multi_region_agg_by_city[(fut_TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]['median_annual_median_loading_rcp45hotter_2030_2059'] - transformers_dict_summary_multi_region_agg_by_city[(fut_TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]['median_annual_median_loading_historical_1990_2019']
    lines_dict_summary_multi_region_agg_by_city[(fut_TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]['diff_in_median_loading_future_minus_baseline'] = lines_dict_summary_multi_region_agg_by_city[(fut_TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]['median_annual_median_loading_rcp45hotter_2030_2059'] - lines_dict_summary_multi_region_agg_by_city[(fut_TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]['median_annual_median_loading_historical_1990_2019']
    # Add column with difference in max loading          
    transformers_dict_summary_multi_region_agg_by_city[(fut_TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]['diff_in_max_loading_future_minus_baseline'] = transformers_dict_summary_multi_region_agg_by_city[(fut_TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]['median_annual_max_loading_rcp45hotter_2030_2059'] - transformers_dict_summary_multi_region_agg_by_city[(fut_TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]['median_annual_max_loading_historical_1990_2019']
    lines_dict_summary_multi_region_agg_by_city[(fut_TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]['diff_in_max_loading_future_minus_baseline'] = lines_dict_summary_multi_region_agg_by_city[(fut_TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]['median_annual_max_loading_rcp45hotter_2030_2059'] - lines_dict_summary_multi_region_agg_by_city[(fut_TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]['median_annual_max_loading_historical_1990_2019']
    
# Create subsets of assets already at or above the historical at-risk threshold.
transformers_dict_summary_multi_region_agg_by_city_at_risk = copy.deepcopy(transformers_dict_summary_multi_region_agg_by_city)
lines_dict_summary_multi_region_agg_by_city_at_risk = copy.deepcopy(lines_dict_summary_multi_region_agg_by_city)

for city in cities:
    df = transformers_dict_summary_multi_region_agg_by_city[(fut_TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]
    # Keep only rows where the value in 'at_risk_filter_column' is above or equal to the threshold
    transformers_dict_summary_multi_region_agg_by_city_at_risk[(fut_TGW_weather_year, TGW_scenario)][(smart_ds_year, city)] =  df[df[at_risk_filter_column] >= xfer_at_risk_threshold] 
    
    df = lines_dict_summary_multi_region_agg_by_city[(fut_TGW_weather_year, TGW_scenario)][(smart_ds_year, city)]
    # Keep only rows where the value in 'at_risk_filter_column' is above or equal to the threshold
    lines_dict_summary_multi_region_agg_by_city_at_risk[(fut_TGW_weather_year, TGW_scenario)][(smart_ds_year, city)] =  df[df[at_risk_filter_column] >= lines_at_risk_threshold] 

## Count number of assets

In [ ]:
# Count transformers and power lines per city

# Get counts
transformer_counts = count_assets_by_city(
    transformers_dict_summary_multi_region_agg_by_city,
    cities
)

line_counts = count_assets_by_city(
    lines_dict_summary_multi_region_agg_by_city,
    cities
)


# Print counts
print("Number of assets per city:")
for city in cities:
    print(
        f"{city}: "
        f"{transformer_counts[city]:,} transformers, "
        f"{line_counts[city]:,} power lines"
    )


# Set KDE maximum high enough to include every asset
max_assets_across_cities = max(
    max(transformer_counts.values()),
    max(line_counts.values())
)

print(f"\nmax_assets_across_cities = {max_assets_across_cities:,}")

## Calculate the share of assets with lower future than historical loading

In [ ]:
# ======================================================================
# Percentage of assets with lower future values than historical values
# ======================================================================

col_b = "median_annual_max_loading_historical_1990_2019"
col_f = "median_annual_max_loading_rcp45hotter_2030_2059"

cities = ("SFO", "GSO", "AUS")

results = []

for city in cities:

    for asset_type, data_dict in [
        ("Transformers", transformers_dict_summary_multi_region_agg_by_city),
        ("Power lines", lines_dict_summary_multi_region_agg_by_city),
    ]:

        # Find the city-level dataframe in the nested dictionary
        city_df = None

        for outer_key, inner_dict in data_dict.items():
            for inner_key, df in inner_dict.items():

                # Expected city-level key structure:
                # (smart_ds_year, city)
                if isinstance(inner_key, tuple) and city in inner_key:
                    city_df = df.copy()
                    break

            if city_df is not None:
                break

        if city_df is None:
            raise KeyError(
                f"Could not find data for {city} - {asset_type}"
            )

        # Keep only assets with valid historical and future values
        valid = city_df[[col_b, col_f]].dropna()

        # Future value lower than historical value
        n_reduced = (valid[col_f] < valid[col_b]).sum()
        n_total = len(valid)

        pct_reduced = (
            100 * n_reduced / n_total
            if n_total > 0
            else np.nan
        )

        results.append({
            "City": city,
            "Asset type": asset_type,
            "N assets": n_total,
            "N reduced": n_reduced,
            "Reduced [%]": pct_reduced,
        })


# ======================================================================
# Create and display results dataframe
# ======================================================================

reduction_df = pd.DataFrame(results)

display(reduction_df)

print("\nPercentage of assets with lower future values than historical values:\n")

for _, row in reduction_df.iterrows():
    print(
        f"{row['City']} - {row['Asset type']}: "
        f"{row['Reduced [%]']:.1f}% "
        f"({row['N reduced']}/{row['N assets']} assets)"
    )

## Scatter plot - median annual max loading

### Full population

#### Scatter + distributions

In [ ]:
start_time = time.time()


fontsize = 12
plt.rcParams.update({
    "font.size": fontsize,
    "axes.labelsize": fontsize,
    "axes.titlesize": fontsize,
    "xtick.labelsize": fontsize,
    "ytick.labelsize": fontsize,
    "legend.fontsize": fontsize,
})

summary_df = plot_diff_scatter_3x2_max_with_marginals_cityagg(
    transformers_dict_summary_multi_region_agg_by_city,
    lines_dict_summary_multi_region_agg_by_city,
    col_b = "median_annual_max_loading_historical_1990_2019",
    col_f = "median_annual_max_loading_rcp45hotter_2030_2059",
    cities=("SFO", "GSO", "AUS"),  # <-- order of cities to display
    baseline_filter_col=at_risk_filter_column,
    filter_threshold_transformers=0,
    filter_threshold_lines=0,
    clip_outliers=False,
    figsize=(6, 8),
    x_lim=(-10, 150),
    y_lim=(-5, 20),
    point_size=3,
    point_alpha=0.05,
    scatter_max_points=max_assets_across_cities,
    show_marginals=True,
    kde_max_points=max_assets_across_cities,
    city_colors=city_colors,
    savepath=fig_scatter_name_max_full_pop, 
    # --- Contour parameters ---
    show_contours=True,
    contour_probs=(0.90,),           # “core” and “outer” regions
    contour_grid_points=120,           
    contour_max_points=max_assets_across_cities,       # subsample for KDE computation
    contour_bandwidth="silverman",   # or a float (e.g., 0.25) to tweak smoothing
    contour_scale="range",           # "range" (uses plot limits) or "std"
    contour_lw=1.8,
    contour_alpha=0.8,
)

end_time = time.time(); print("Runtime:", (end_time - start_time) / 60, "minutes")

#### Tables with values from scatter plot (median, (5th,95th))

In [ ]:
# ======================================================================
# Format scatter-plot summary statistics for display and LaTeX
# ======================================================================

table_df = summary_df.copy()

# Preserve the desired row order, matching the figure
city_order = {
    "SFO": 0,
    "GSO": 1,
    "AUS": 2,
}

asset_order = {
    "Transformers": 0,
    "Power lines": 1,
}

table_df["_city_order"] = table_df["City"].map(city_order)
table_df["_asset_order"] = table_df["Asset type"].map(asset_order)

table_df = table_df.sort_values(
    by=["_city_order", "_asset_order"]
).reset_index(drop=True)

# Replace city abbreviations with full names
table_df["City"] = table_df["City"].map(
    lambda city: CITY_MAP.get(city, city)
)

# Format median and percentiles
decimals = 1

table_df["Future loading, median (5th, 95th) [%]"] = table_df.apply(
    lambda row: (
        f"{row['Future loading median [%]']:.{decimals}f} "
        f"({row['Future loading 5th percentile [%]']:.{decimals}f}, "
        f"{row['Future loading 95th percentile [%]']:.{decimals}f})"
        if row["N"] > 0
        else "—"
    ),
    axis=1,
)

table_df["Change from historical, median (5th, 95th) [percentage points]"] = table_df.apply(
    lambda row: (
        f"{row['Change median [percentage points]']:.{decimals}f} "
        f"({row['Change 5th percentile [percentage points]']:.{decimals}f}, "
        f"{row['Change 95th percentile [percentage points]']:.{decimals}f})"
        if row["N"] > 0
        else "—"
    ),
    axis=1,
)

# Retain only the manuscript-table columns
table_df = table_df[
    [
        "City",
        "Asset type",
        "N",
        "Future loading, median (5th, 95th) [%]",
        "Change from historical, median (5th, 95th) [percentage points]",
    ]
]


# ======================================================================
# Display table without repeated city labels
# ======================================================================

display_table = table_df.copy()

# Show the city name only in the first row of each city group
display_table.loc[
    display_table["City"].duplicated(),
    "City",
] = ""

display(display_table)


# ======================================================================
# Create LaTeX table
# ======================================================================

latex_df = table_df.copy()

# Use shorter, LaTeX-safe column labels
latex_df = latex_df.rename(
    columns={
        "Future loading, median (5th, 95th) [%]":
            r"Future loading [\%], median (5th, 95th)",
        "Change from historical, median (5th, 95th) [percentage points]":
            r"Change from historical [percentage points], median (5th, 95th)",
    }
)

# Use a hierarchical index so repeated city labels are combined using \multirow
latex_df = latex_df.set_index(
    [
        "City",
        "Asset type",
    ]
)

latex_table = latex_df.to_latex(
    index=True,
    escape=False,
    sparsify=True,
    multirow=True,
    multicolumn=False,
    na_rep="--",
    column_format="llcrr",
    position="htbp",
    caption=(
        "caption"
    ),
    label="tab:future_loading_change_summary",
)

print(latex_table)

### At-risk population

#### Scatter plot

In [ ]:
start_time = time.time()


fontsize = 12
plt.rcParams.update({
    "font.size": fontsize,
    "axes.labelsize": fontsize,
    "axes.titlesize": fontsize,
    "xtick.labelsize": fontsize,
    "ytick.labelsize": fontsize,
    "legend.fontsize": fontsize,
})

summary_df = plot_diff_scatter_3x2_max_with_marginals_cityagg(
    transformers_dict_summary_multi_region_agg_by_city,
    lines_dict_summary_multi_region_agg_by_city,
    col_b = "median_annual_max_loading_historical_1990_2019",
    col_f = "median_annual_max_loading_rcp45hotter_2030_2059",
    cities=("SFO", "GSO", "AUS"),  # <-- order of cities to display
    baseline_filter_col=at_risk_filter_column,
    filter_threshold_transformers=xfer_at_risk_threshold,
    filter_threshold_lines=lines_at_risk_threshold,
    clip_outliers=False,
    figsize=(6, 8),
    x_lim=(40, 160),
    y_lim=(-5, 20),
    point_size=3,
    point_alpha=0.05,
    scatter_max_points=max_assets_across_cities,
    show_marginals=True,
    kde_max_points=max_assets_across_cities,
    city_colors=city_colors,
    savepath=fig_scatter_name_max_at_risk_pop, 
    show_contours=True,
    contour_probs=(0.90,),           # “core” and “outer” regions
    contour_grid_points=120,           
    contour_max_points=max_assets_across_cities,       # subsample for KDE computation
    contour_bandwidth="silverman",   # or a float (e.g., 0.25) to tweak smoothing
    contour_scale="range",           # "range" (uses plot limits) or "std"
    contour_lw=1.8,
    contour_alpha=0.8,
    transformer_safe_label_x=65,
    transformer_at_risk_label_x=97,
    transformer_critical_label_x=130,
    line_safe_label_x=65,
    line_at_risk_label_x=97,
    line_critical_label_x=130,
)


end_time = time.time(); print("Runtime:", (end_time - start_time) / 60, "minutes")

#### Table

In [ ]:
# ======================================================================
# Format scatter-plot summary statistics for display and LaTeX
# ======================================================================

table_df = summary_df.copy()

# Preserve the desired row order, matching the figure
city_order = {
    "SFO": 0,
    "GSO": 1,
    "AUS": 2,
}

asset_order = {
    "Transformers": 0,
    "Power lines": 1,
}

table_df["_city_order"] = table_df["City"].map(city_order)
table_df["_asset_order"] = table_df["Asset type"].map(asset_order)

table_df = table_df.sort_values(
    by=["_city_order", "_asset_order"]
).reset_index(drop=True)

# Replace city abbreviations with full names
table_df["City"] = table_df["City"].map(
    lambda city: CITY_MAP.get(city, city)
)

# Format median and percentiles
decimals = 1

table_df["Future loading, median (5th, 95th) [%]"] = table_df.apply(
    lambda row: (
        f"{row['Future loading median [%]']:.{decimals}f} "
        f"({row['Future loading 5th percentile [%]']:.{decimals}f}, "
        f"{row['Future loading 95th percentile [%]']:.{decimals}f})"
        if row["N"] > 0
        else "—"
    ),
    axis=1,
)

table_df["Change from historical, median (5th, 95th) [percentage points]"] = table_df.apply(
    lambda row: (
        f"{row['Change median [percentage points]']:.{decimals}f} "
        f"({row['Change 5th percentile [percentage points]']:.{decimals}f}, "
        f"{row['Change 95th percentile [percentage points]']:.{decimals}f})"
        if row["N"] > 0
        else "—"
    ),
    axis=1,
)

# Retain only the manuscript-table columns
table_df = table_df[
    [
        "City",
        "Asset type",
        "N",
        "Future loading, median (5th, 95th) [%]",
        "Change from historical, median (5th, 95th) [percentage points]",
    ]
]


# ======================================================================
# Display table without repeated city labels
# ======================================================================

display_table = table_df.copy()

# Show the city name only in the first row of each city group
display_table.loc[
    display_table["City"].duplicated(),
    "City",
] = ""

display(display_table)


# ======================================================================
# Create publication-ready LaTeX table
# ======================================================================

latex_df = table_df.copy()

# Use shorter, LaTeX-safe column labels
latex_df = latex_df.rename(
    columns={
        "Future loading, median (5th, 95th) [%]":
            r"Future loading [\%], median (5th, 95th)",
        "Change from historical, median (5th, 95th) [percentage points]":
            r"Change from historical [percentage points], median (5th, 95th)",
    }
)

# Use a hierarchical index so repeated city labels are combined using \multirow
latex_df = latex_df.set_index(
    [
        "City",
        "Asset type",
    ]
)

latex_table = latex_df.to_latex(
    index=True,
    escape=False,
    sparsify=True,
    multirow=True,
    multicolumn=False,
    na_rep="--",
    column_format="llcrr",
    position="htbp",
    caption=(
        "caption"
    ),
    label="tab:future_loading_change_summary",
)

print(latex_table)